`https://drive.google.com/file/d 10dPRAt9ZlE8OGoBmkpplFeKSgNF_44x0/view?usp=sharing`

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import kagglehub
adhamashraf202200953_technical_parsed_questions_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')

print('Data source import complete.')


Using Colab cache for faster access to the 'technical-parsed-questions' dataset.
Data source import complete.


In [ ]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
# !pip install --upgrade torchao

In [ ]:
import gc
import torch
import os
import json
import glob
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
MODEL_NAME = "facebook/bart-large"
DATA_FOLDER = "/root/.cache/kagglehub/datasets/adhamashraf202200953/technical-parsed-questions/versions/1"
OUTPUT_DIR = "./outputs/STABLE_BART"

BATCH_SIZE = 4             # Bumped back up to 4 for faster training on BART
GRADIENT_ACC_STEPS = 8     # Reset to 8 to maintain effective batch size of 32
MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 256

gc.collect()
torch.cuda.empty_cache()

In [ ]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        if record.get("type") == "essay" and record.get("model_answer"):
                            all_essay_records.append(record)
                    except: continue
    return all_essay_records

def preprocess_function(examples):
    inputs = [c for c in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)

    targets = [f"Scenario: {s} Question: {q} Answer: {a}"
               for s, q, a in zip(examples["scenario_context"], examples["question"], examples["model_answer"])]

    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)


In [ ]:
raw_essay_data = migrate_essay_data(DATA_FOLDER)
dataset = Dataset.from_list(raw_essay_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float32
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.1,
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, peft_config)

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

In [ ]:
print("Total training samples:", len(split_dataset["train"]))

Total training samples: 1988


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=5,
    learning_rate=2e-4,          # Stable sweet spot for generative LoRA fine-tuning
    num_train_epochs=3,
    max_grad_norm=1.0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    optim="adamw_torch",
    report_to="none"
)

In [ ]:
# Initialize Metrics
rouge_metric = evaluate.load("rouge")


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100
    ),
    compute_metrics=compute_metrics
)


trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
50,20.407764,2.197958,0.393011,0.114111,0.233975,0.234126
100,19.325316,2.094932,0.398448,0.117372,0.241404,0.241390
150,18.516104,2.055882,0.398021,0.115318,0.241608,0.241731
189,18.177325,2.046493,0.399098,0.117393,0.243822,0.244054


TrainOutput(global_step=189, training_loss=20.032348370425915, metrics={'train_runtime': 2872.2893, 'train_samples_per_second': 2.076, 'train_steps_per_second': 0.066, 'total_flos': 4455773896704000.0, 'train_loss': 20.032348370425915, 'epoch': 3.0})

In [ ]:
!zip -r "/content/file.zip" "/content/outputs"

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/STABLE_BART/ (stored 0%)
  adding: content/outputs/STABLE_BART/checkpoint-189/ (stored 0%)
  adding: content/outputs/STABLE_BART/checkpoint-189/training_args.bin (deflated 54%)
  adding: content/outputs/STABLE_BART/checkpoint-189/rng_state.pth (deflated 26%)
  adding: content/outputs/STABLE_BART/checkpoint-189/adapter_config.json (deflated 59%)
  adding: content/outputs/STABLE_BART/checkpoint-189/trainer_state.json (deflated 74%)
  adding: content/outputs/STABLE_BART/checkpoint-189/scheduler.pt (deflated 62%)
  adding: content/outputs/STABLE_BART/checkpoint-189/README.md (deflated 66%)
  adding: content/outputs/STABLE_BART/checkpoint-189/tokenizer_config.json (deflated 50%)
  adding: content/outputs/STABLE_BART/checkpoint-189/adapter_model.safetensors (deflated 7%)
  adding: content/outputs/STABLE_BART/checkpoint-189/tokenizer.json (deflated 82%)
  adding: content/outputs/STABLE_BART/checkpoint-189/optimizer.pt (deflated 

In [ ]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

BASE_MODEL = "facebook/bart-large"
BART_LORA_DIR = "/content/outputs/STABLE_BART/checkpoint-189"

print("Loading BART Large...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float32
)
model = PeftModel.from_pretrained(base_model, BART_LORA_DIR)

# def ask_bart(context_text):
#     inputs = tokenizer(context_text, return_tensors="pt", max_length=384, truncation=True).to(model.device)
#     outputs = model.generate(**inputs, max_length=256, num_beams=4, early_stopping=True)
#     return tokenizer.decode(outputs[0], skip_special_tokens=True)

def ask_bart(context_text):
    inputs = tokenizer(context_text, return_tensors="pt", max_length=384, truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=256,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,  # <-- Prevents the model from repeating the same 3-word phrases
        temperature=0.7,         # <-- Adds a tiny bit of creative variety to make it sound more human
        do_sample=True           # <-- Required for temperature to work
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading BART Large...


Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

In [ ]:
sample_context = """
When a Docker container exits immediately with a status code 0, it typically means the primary
foreground process specified in the ENTRYPOINT or CMD has completed its execution. Containers
require a persistent foreground process to remain active. To keep a debugging container running
without an active service, developers often append a blocking command like 'tail -f /dev/null'
or run the container interactively using the '-it' flags, though this can consume unnecessary
idle system resources.
"""
print(ask_bart(sample_context))

Scenario: Your team is experiencing issues with persistent foreground processes in your Docker container. You need to decide whether to implement a blocking command or interactively run the container interactively using the '-it' flags. Question: Propose an alternative approach to managing persistent foreground process in your container. Explain the tradeoffs involved in this approach. Answer: Implement a blocking process like 'tail -f /dev/null' or 'tail-f /usr/local/bin'. This allows the container to run interactively without consuming unnecessary system resources. However, this approach may introduce additional overhead due to the need for persistent processes.


In [ ]:
sample_context_2 = """
When a Linux kernel update occurs, proprietary graphics drivers often break because the pre-compiled
kernel module no longer matches the newly installed kernel interface. To prevent this complete loss
of GUI display after an update, systems use DKMS (Dynamic Kernel Module Support). DKMS automatically
recompiles the proprietary driver's source code against the headers of the new kernel during the
boot process, slightly increasing overall update installation time but ensuring video stability.
"""
print(ask_bart(sample_context_2))

Scenario: Your company's high-end e-commerce platform currently uses a proprietary graphics driver, but you're experiencing issues with graphical performance after a kernel update. Question: Explain how you would implement a dynamic kernel module support (DKMS) to prevent graphical performance degradation during kernel updates. Explain the tradeoffs involved in this approach. Answer: Use DKMS to automatically recompile the proprietary driver's source code against the headers of the new kernel during the boot process, reducing overall update installation time but ensuring video stability. However, this may lead to increased memory usage due to the need to recompile the source code.
